# ViT-Motion extras — before/after Grad-CAM & λ sweep

Two follow-ups to `kaggle_interpret_plus`:
1. **Before/after Grad-CAM** — the strongest visual: attention moving sky→ground.
2. **λ sweep** — the accuracy vs. focus trade-off of the attention penalty.

Attach the same 3 inputs (code + data + artifacts). GPU + Internet ON.


In [ ]:
# setup (robust): stage code, deps, locate data/checkpoint, build/repair manifest
import os, sys, glob, shutil, pathlib
print('Attached inputs:')
for d in sorted(glob.glob('/kaggle/input/*')): print('   ', d)

codes = glob.glob('/kaggle/input/**/inspect_dataset.py', recursive=True)
assert codes, 'CODE dataset not attached (no inspect_dataset.py). Add Input -> your vit-motion-code.'
CODE_SRC = str(pathlib.Path(sorted(codes, key=len)[0]).parent)
WORK = '/kaggle/working/vit_motion_project'
if os.path.exists(WORK): shutil.rmtree(WORK)
shutil.copytree(CODE_SRC, WORK); os.chdir(WORK); sys.path.insert(0, WORK)
get_ipython().system("pip -q install 'timm>=1.0' 'opencv-python>=4.9' >/dev/null")

csvs = [c for c in glob.glob('/kaggle/input/**/samples.csv', recursive=True) if not c.startswith(CODE_SRC)]
assert csvs, ('DATA dataset not attached. Add Input -> your vit-motion-dataset '
              '(or the build notebooks output) so samples.csv is visible under /kaggle/input.')
DATA_ROOT = sorted({str(pathlib.Path(c).parent.parent) for c in csvs}, key=len)[0]

cks = glob.glob('/kaggle/input/**/best.pt', recursive=True)
assert cks, 'ARTIFACTS dataset not attached (no best.pt). Add Input -> your vit-motion-artifacts.'
CKPT_SRC = sorted(cks, key=len)[0]
print('DATA_ROOT :', DATA_ROOT); print('checkpoint:', CKPT_SRC)

import yaml
yaml.safe_dump(yaml.safe_load(open('config_kaggle.yaml')), open('config_run.yaml','w'), sort_keys=False, allow_unicode=True)
os.makedirs('artifacts/manifest', exist_ok=True)

# Reuse the prebuilt manifest ONLY if its absolute RGB paths resolve to the attached
# data (they were baked at build time). Otherwise rebuild so paths match this session.
import pandas as pd
man = glob.glob('/kaggle/input/**/manifest/manifest.csv', recursive=True)
reuse = False
if man:
    mm = pd.read_csv(man[0])
    if 'rgb_abs_path' in mm and os.path.exists(str(mm['rgb_abs_path'].iloc[0])):
        src = str(pathlib.Path(man[0]).parent)
        for f in ['manifest.csv','normalization.json','splits.json']:
            if os.path.exists(os.path.join(src,f)): shutil.copy(os.path.join(src,f), 'artifacts/manifest/'+f)
        reuse = True; print('reused prebuilt manifest from', src)
if not reuse:
    print('rebuilding manifest for the attached data root (~5-6 min)...')
    get_ipython().system('python inspect_dataset.py --config config_run.yaml --data-root "{DATA_ROOT}"')

CKPT = 'artifacts/runs/vit_motion_temporal_cr_v0_2_1/best.pt'
os.makedirs(os.path.dirname(CKPT), exist_ok=True); shutil.copy(CKPT_SRC, CKPT)
m = pd.read_csv('artifacts/manifest/manifest.csv')
pool = m[m.split=='test'] if (m.split=='test').any() else m
EXP = sorted(pool.experiment_id.astype(str).unique())[0]
print('experiment:', EXP)

In [ ]:
# make sure we have a fine-tuned checkpoint (reuse if the artifacts dataset already has it)
RRR='artifacts/runs/vit_motion_rrr/best_rrr.pt'
found=glob.glob('/kaggle/input/**/best_rrr.pt',recursive=True)
os.makedirs(os.path.dirname(RRR),exist_ok=True)
if found:
    shutil.copy(sorted(found,key=len)[0],RRR); print('reused',found[0])
else:
    get_ipython().system('python finetune_rrr.py --config config_run.yaml --checkpoint "{CKPT}" --epochs 3 --lambda-rrr 0.5 --horizon-frac 0.5 --target norm --max-steps 150 --output "{RRR}"')

## 1) Before / after Grad-CAM (sky → ground)


In [ ]:
!python compare_interpret.py --config config_run.yaml \
  --checkpoint-baseline "{CKPT}" --checkpoint-finetuned "{RRR}" \
  --experiment "{EXP}" --num-samples 4 --target yaw \
  --output artifacts/interpretability/before_after.png
from IPython.display import Image, display
display(Image('artifacts/interpretability/before_after.png'))

## 2) λ sweep — focus vs. accuracy trade-off

For each penalty weight λ we fine-tune from `best.pt`, then measure sky-saliency and R².
λ=0 is the control (pure task fine-tune). This is compute-heavy — trim `LAMBDAS` or
`--max-steps` if you are short on GPU time.


In [ ]:
import subprocess, json, os
LAMBDAS=[0.0, 0.1, 0.5, 1.0, 2.0]
rows=[]
for lam in LAMBDAS:
    ck=f'artifacts/runs/sweep/lam_{lam}.pt'
    os.makedirs(os.path.dirname(ck),exist_ok=True)
    subprocess.run(f'python finetune_rrr.py --config config_run.yaml --checkpoint "{CKPT}" '
                   f'--epochs 2 --lambda-rrr {lam} --horizon-frac 0.5 --target norm --max-steps 120 '
                   f'--output "{ck}"',shell=True,check=True)
    subprocess.run(f'python interpret_quantify.py --config config_run.yaml --checkpoint "{ck}" '
                   f'--experiments auto --max-exp 4 --num-per-exp 30 --target yaw --tag lam_{lam}',shell=True,check=True)
    subprocess.run(f'python evaluate_experiment.py --config config_run.yaml --checkpoint "{ck}" '
                   f'--experiment "{EXP}" --output-dir artifacts/eval_sweep_{lam}',shell=True,check=True)
    q=json.load(open(f'artifacts/quantify/sky_ground_lam_{lam}.json'))
    e=json.load(open(f'artifacts/eval_sweep_{lam}/{EXP}/metrics.json'))['outputs']
    rows.append({'lam':lam,'sky':q['sky_saliency_fraction_mean'],
                 'r2_dx':e['next_body_dx']['r2'],'r2_yaw':e['next_delta_yaw']['r2']})
    print(rows[-1])

In [ ]:
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
lam=[r['lam'] for r in rows]; sky=[r['sky'] for r in rows]
r2y=[r['r2_yaw'] for r in rows]; r2x=[r['r2_dx'] for r in rows]
fig,ax1=plt.subplots(figsize=(8,5),constrained_layout=True)
ax1.plot(lam,sky,'o-',color='#B3261E',label='sky saliency fraction'); ax1.set_xlabel('penalty weight  λ')
ax1.set_ylabel('sky saliency fraction',color='#B3261E'); ax1.axhline(0.5,ls='--',color='#999',lw=1)
ax2=ax1.twinx(); ax2.plot(lam,r2y,'s-',color='#1B7A3D',label='yaw R²'); ax2.plot(lam,r2x,'^-',color='#065A82',label='dx R²')
ax2.set_ylabel('R² (dx, yaw)')
l1,la1=ax1.get_legend_handles_labels(); l2,la2=ax2.get_legend_handles_labels(); ax1.legend(l1+l2,la1+la2,loc='center right')
ax1.set_title('Attention-guidance trade-off: focus vs. accuracy')
fig.savefig('artifacts/quantify/lambda_sweep.png',dpi=170); print('saved lambda_sweep.png')
from IPython.display import Image, display; display(Image('artifacts/quantify/lambda_sweep.png'))

In [ ]:
import shutil; shutil.make_archive('/kaggle/working/vit_motion_extras','zip','artifacts'); print('done -> vit_motion_extras.zip')